In [1]:
#import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
#check your notebook location
import os

print(os.getcwd())

C:\Users\Lenovo\Desktop\Mahak Project\Ecommerce-Sales-Profit-Analytics\python\notebooks


In [10]:
#load the three datasets
orders = pd.read_csv(
    "../../data/raw/olist_orders_dataset.csv"
)

order_items = pd.read_csv(
    "../../data/raw/olist_order_items_dataset.csv"
)

customers = pd.read_csv(
    "../../data/raw/olist_customers_dataset.csv"
)

In [11]:
# Confirm they loaded
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Customers:", customers.shape)

Orders: (99441, 8)
Order Items: (112650, 7)
Customers: (99441, 5)


In [5]:
# Inspect each dataset
orders.head()
order_items.head()
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [6]:
## Check data types
print("ORDERS")
print(orders.dtypes)

print("\nORDER ITEMS")
print(order_items.dtypes)

print("\nCUSTOMERS")
print(customers.dtypes)

ORDERS
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

ORDER ITEMS
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

CUSTOMERS
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object


In [7]:
## Check missing value
print("Orders missing values:")
print(orders.isnull().sum())

print("\nOrder Items missing values:")
print(order_items.isnull().sum())

print("\nCustomers missing values:")
print(customers.isnull().sum())

Orders missing values:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order Items missing values:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Customers missing values:
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [9]:
## Check duplicate rows
print("Orders duplicate rows:", orders.duplicated().sum())
print("Order Items duplicate rows:", order_items.duplicated().sum())
print("Customers duplicate rows:", customers.duplicated().sum())

Orders duplicate rows: 0
Order Items duplicate rows: 0
Customers duplicate rows: 0


In [10]:
## Create one validation  summary
validation_summary = pd.DataFrame({
    "Dataset": ["Orders", "Order Items", "Customers"],
    "Rows": [
        orders.shape[0],
        order_items.shape[0],
        customers.shape[0]
    ],
    "Columns": [
        orders.shape[1],
        order_items.shape[1],
        customers.shape[1]
    ],
    "MissingValues": [
        orders.isnull().sum().sum(),
        order_items.isnull().sum().sum(),
        customers.isnull().sum().sum()
    ],
    "DuplicateRows": [
        orders.duplicated().sum(),
        order_items.duplicated().sum(),
        customers.duplicated().sum()
    ]
})

validation_summary

,Dataset,Rows,Columns,MissingValues,DuplicateRows
0,Orders,99441,8,4908,0
1,Order Items,112650,7,0,0
2,Customers,99441,5,0,0


In [14]:
# P2: Validate Date Columns with Pandas

date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

date_report = []

for col in date_columns:

    # Convert the column without changing the original DataFrame
    converted = pd.to_datetime(
        orders[col],
        errors='coerce'
    )

    # Values that originally existed but failed conversion
    invalid_dates = (
        orders[col].notna() & converted.isna()
    ).sum()

    date_report.append({
        'Column': col,
        'MissingValues': orders[col].isna().sum(),
        'InvalidDates': invalid_dates,
        'MinDate': converted.min(),
        'MaxDate': converted.max()
    })

date_report = pd.DataFrame(date_report)

date_report


    

,Column,MissingValues,InvalidDates,MinDate,MaxDate
0,order_purchase_timestamp,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,order_approved_at,160,0,2016-09-15 12:16:38,2018-09-03 17:40:06
2,order_delivered_carrier_date,1783,0,2016-10-08 10:34:01,2018-09-11 19:48:28
3,order_delivered_customer_date,2965,0,2016-10-11 13:46:32,2018-10-17 13:22:46
4,order_estimated_delivery_date,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00


In [13]:
## Calculate the overall purchase _date range
purchase_dates = pd.to_datetime(
    orders['order_purchase_timestamp'],
    errors='coerce'
)

print("Earliest Purchase:", purchase_dates.min())
print("Latest Purchase:", purchase_dates.max())

Earliest Purchase: 2016-09-04 21:15:19
Latest Purchase: 2018-10-17 17:30:18


In [20]:
# P3: Validate Numeric Columns

import pandas as pd

numeric_columns = [
    'price',
    'freight_value'
]

numeric_report = []

for col in numeric_columns:

    # Convert column to numeric
    converted = pd.to_numeric(
        order_items[col],
        errors='coerce'
    )

    numeric_report.append({
        'Column': col,
        'MissingValues': converted.isna().sum(),
        'NegativeValues': (converted < 0).sum(),
        'ZeroValues': (converted == 0).sum(),
        'Minimum': converted.min(),
        'Maximum': converted.max(),
        'Average': converted.mean()
    })

numeric_report = pd.DataFrame(numeric_report)

numeric_report

,Column,MissingValues,NegativeValues,ZeroValues,Minimum,Maximum,Average
0,price,0,0,0,0.85,6735.00,120.653739
1,freight_value,0,0,383,0.00,409.68,19.990320


In [15]:
# P4: Validate Categorical Columns

import pandas as pd

category_checks = [
    ('order_status', orders),
    ('customer_state', customers)
]

category_report = []

for col, df in category_checks:

    # Convert values to text and remove extra spaces
    cleaned = df[col].astype('string').str.strip()

    category_report.append({
        'Column': col,
        'MissingValues': cleaned.isna().sum(),
        'UniqueValues': cleaned.nunique(dropna=True),
        'BlankValues': cleaned.eq('').sum()
    })

category_report = pd.DataFrame(category_report)

category_report

,Column,MissingValues,UniqueValues,BlankValues
0,order_status,0,8,0
1,customer_state,0,27,0


In [16]:
# Show the top 10 values
for col, df in category_checks:

    print(f"\nTop 10 values for: {col}")
    print(
        df[col]
        .astype('string')
        .str.strip()
        .value_counts(dropna=False)
        .head(10)
    )


Top 10 values for: order_status
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: Int64

Top 10 values for: customer_state
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
Name: count, dtype: Int64


In [20]:
# P5: Validate Key Uniqueness and Duplicates

import pandas as pd

key_checks = [
    ('order_id', orders),
    ('customer_id', customers),
    ('customer_unique_id', customers),
    ('order_item_id', order_items)
]

uniq_report = []

for col, df in key_checks:

    cleaned = df[col].astype('string').str.strip()

    uniq_report.append({
        'Column': col,
        'TotalRows': len(cleaned),
        'UniqueValues': cleaned.nunique(dropna=True),
        'DuplicateValues': cleaned.duplicated().sum(),
        'MissingValues': cleaned.isna().sum()
    })

uniq_report = pd.DataFrame(uniq_report)

uniq_report

,Column,TotalRows,UniqueValues,DuplicateValues,MissingValues
0,order_id,99441,99441,0,0
1,customer_id,99441,99441,0,0
2,customer_unique_id,99441,96096,3345,0
3,order_item_id,112650,21,112629,0


In [21]:
## Composite duplicates
composite_duplicates = order_items.duplicated(
    subset=['order_id', 'order_item_id']
).sum()

print("Duplicate order_id + order_item_id combinations:",
      composite_duplicates)

Duplicate order_id + order_item_id combinations: 0


In [24]:
# P6: Final Data Validation Summary

import pandas as pd

datasets = [
    ('Orders', orders),
    ('Order Items', order_items),
    ('Customers', customers)
]

summary_report = []

for dataset_name, df in datasets:

    summary_report.append({
        'Dataset': dataset_name,
        'Rows': df.shape[0],
        'Columns': df.shape[1],
        'MissingValues': df.isna().sum().sum(),
        'DuplicateRows': df.duplicated().sum()
    })

summary_report = pd.DataFrame(summary_report)

summary_report

,Dataset,Rows,Columns,MissingValues,DuplicateRows
0,Orders,99441,8,4908,0
1,Order Items,112650,7,0,0
2,Customers,99441,5,0,0


In [25]:
## KEY CHECKS
print(
    "Orders order_id unique:",
    orders['order_id'].is_unique
)

print(
    "Customers customer_id unique:",
    customers['customer_id'].is_unique
)

print(
    "Customers customer_unique_id unique:",
    customers['customer_unique_id'].is_unique
)

print(
    "Order Items order_id + order_item_id unique:",
    ~order_items.duplicated(
        subset=['order_id', 'order_item_id']
    ).any()
)

Orders order_id unique: True
Customers customer_id unique: True
Customers customer_unique_id unique: False
Order Items order_id + order_item_id unique: True


In [ ]:
## Validation #Conclusion

The raw Olist datasets were successfully loaded and validated using Pandas. 
No duplicate rows were found in the main datasets, and the order-item composite key `order_id + order_item_id` is unique. 
The orders dataset contains missing values in several date-related fields, while the order-items and customers datasets have no missing values in the checked fields. 
The data is ready for exploratory data analysis, with missing values and date completeness considered during the EDA stage.